# 2. Loops divided among workers

Most analysis code that takes a long time is a loop: over subjects, over
random seeds, over model settings. If the iterations do not depend on one
another, they can run at the same time. Clustrix works this out from the
function's source and divides the loop for you. On this machine the pieces go
to worker processes; on a cluster they become jobs (tutorials 3 and 4).

This tutorial covers, in order:

1. three realistic loops, timed directly and through `@cluster`;
2. loops that are **not** divided, the reason given for each, and how to
   rewrite them;
3. loops that look divisible but are found unsafe while running;
4. what no check can see;
5. how clustrix decides whether dividing is worth it, printing, exceptions,
   `cores=-1`, and nested calls.

Nothing here needs a cluster. The timings are from the machine that built
this documentation and are there to show the shape of the result, not to be a
benchmark.

In [1]:
import os
import time

import numpy as np

import clustrix
from clustrix import cluster, configure, explain
from clustrix.loop_workers import available_cpus

configure(cluster_type="local")
print("CPUs this process may use:", available_cpus())

CPUs this process may use: 4


A small helper to time a call. It is not part of clustrix.

In [2]:
def timed(label, function, *args):
    started = time.perf_counter()
    result = function(*args)
    print(f"{label:<28} {time.perf_counter() - started:6.2f} s")
    return result

## A CPU-bound loop

Pure-Python arithmetic, one long recurrence per starting value. The function
also records the process ID that handled each iteration, so we can see who did
the work.

In [3]:
def hash_chain(seeds, length):
    finals, pids = {}, set()
    for seed in seeds:
        value = seed
        for step in range(length):
            value = (value * 6364136223846793005 + step) % 18446744073709551557
        finals[seed] = value
        pids.add(os.getpid())
    return finals, sorted(pids)


direct, direct_pids = timed("direct call", hash_chain, list(range(8)), 2_000_000)
print("processes that did the work:", direct_pids, "(this notebook is", os.getpid(), ")")

direct call                    2.60 s
processes that did the work: [8975] (this notebook is 8975 )


Now the same function through `@cluster(cores=4)`. `cluster(...)(function)` is
the decorator written as a call, which lets us keep the plain function for
comparison.

In [4]:
fast_hash_chain = cluster(cores=4)(hash_chain)

divided, worker_pids = timed("@cluster(cores=4), 1st call", fast_hash_chain, list(range(8)), 2_000_000)
print("processes that did the work:", worker_pids)
print("same answer as the direct call:", divided == direct)

@cluster(cores=4), 1st call    1.55 s
processes that did the work: [9001, 9002, 9003, 9004]
same answer as the direct call: True


Four other processes did the work and the answer is identical. The first
call also paid for starting the workers. They are kept for the rest of the
session, so a second call shows what a call costs after that.

In [5]:
divided, worker_pids = timed("@cluster(cores=4), 2nd call", fast_hash_chain, list(range(8)), 2_000_000)
print("processes that did the work:", worker_pids)

@cluster(cores=4), 2nd call    1.05 s
processes that did the work: [9001, 9002, 9003, 9004]


The worker processes are the same ones. `explain` shows why this loop
qualified.

In [6]:
print(explain(hash_chain))

hash_chain:
  line 3: `for seed in seeds` -- SPLIT
      collected in order: finals, pids
      read-only (checked at run time): length
      private to each iteration: seed, step, value
  code before the loop runs in the caller
  code after the loop runs in the caller


Only the outer loop, at the top level of the function, is considered. The
inner `for step in range(length)` belongs to one iteration and stays inside
it. `finals` and `pids` are only written to, `length` and `os` are only read,
and `seed`, `step` and `value` are assigned afresh by every iteration. Nothing
passes from one iteration to the next.

## A seeded simulation

A Monte Carlo estimate of pi with several replicates. Each replicate creates
its own generator from its own seed, so a replicate's numbers do not depend on
which replicates ran before it.

In [7]:
def monte_carlo_pi(replicates, samples):
    estimates = []
    for replicate in range(replicates):
        rng = np.random.default_rng(1000 + replicate)
        points = rng.random((samples, 2))
        inside = (points**2).sum(axis=1) <= 1.0
        estimates.append(4.0 * float(inside.mean()))
    return estimates


direct = timed("direct call", monte_carlo_pi, 8, 4_000_000)
divided = timed("@cluster(cores=4)", cluster(cores=4)(monte_carlo_pi), 8, 4_000_000)

print("estimates:", [round(e, 5) for e in divided])
print("identical, bit for bit:", divided == direct)

direct call                    1.01 s


@cluster(cores=4)              0.57 s
estimates: [3.1413, 3.14268, 3.14239, 3.14158, 3.14086, 3.14139, 3.14008, 3.14163]
identical, bit for bit: True


The results are equal exactly, not approximately. The iterations do not write
to `estimates` themselves; each records what it would have appended, and the
records are applied in iteration order afterwards. That also holds for sums:
`total += x` in a divided loop adds the same numbers in the same order as a
plain run, so floating-point results do not change.

## A model sweep

Fit one model per setting and keep the best. The data is prepared once, before
the loop; every iteration reads it.

In [8]:
def model_sweep(depths):
    from sklearn.datasets import make_classification
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score

    features, labels = make_classification(
        n_samples=3000, n_features=30, n_informative=12, random_state=0
    )
    scores = {}
    for depth in depths:
        model = RandomForestClassifier(n_estimators=60, max_depth=depth, random_state=0)
        scores[depth] = float(cross_val_score(model, features, labels, cv=3).mean())
    best = max(scores, key=scores.get)
    return best, scores


print(explain(model_sweep))

model_sweep:
  line 10: `for depth in depths` -- SPLIT
      collected in order: scores
      read-only (checked at run time): RandomForestClassifier, cross_val_score, features, labels
      private to each iteration: depth, model
  code before the loop runs as its own job
  code after the loop runs in the caller


`features` and `labels` are listed as read-only: they are built once and sent
to each worker. The line `best = max(...)` comes after the loop and runs once
on the collected scores, so "return the best configuration" needs nothing
special.

The last two lines of the report are about where the code around the loop
runs. On this machine it always runs in this process. On a cluster, the code
before this loop would be a job of its own, because building the data is real
work; tutorial 4 shows that.

In [9]:
depths = [2, 4, 6, 8, 10, 12, 16, None]

direct = timed("direct call", model_sweep, depths)
divided = timed("@cluster(cores=4)", cluster(cores=4)(model_sweep), depths)

best, scores = divided
print("best max_depth:", best, "with accuracy", round(scores[best], 4))
print("same as the direct call:", divided == direct)

direct call                    8.04 s


@cluster(cores=4)              3.66 s
best max_depth: 16 with accuracy 0.9093
same as the direct call: True


A note on library threads. NumPy, SciPy and scikit-learn can start threads of
their own, as many as the machine has CPUs. Four workers each doing that would
compete for the same four CPUs, so clustrix gives each worker its share (four
CPUs among four workers is one thread each) through `OMP_NUM_THREADS` and the
related variables. If you set one of those variables yourself before starting
Python, the workers inherit your value unchanged. When you compare timings for
code that uses multi-threaded linear algebra, remember that the direct call
was using the library's default.

## Loops that are not divided, and why

Clustrix divides a loop only when every name the loop body touches is one of
three kinds: private to an iteration, read-only, or a collector that is only
written to. Anything else is refused, the function runs as one piece, and the
result is still correct. `explain` gives the line and the reason. The five
cases below are the ones you will meet, each followed by a rewrite.

### A value carried from one iteration to the next

In [10]:
def best_score(candidates):
    best = None
    for candidate in candidates:
        score = -(candidate - 3.2) ** 2
        if best is None or score > best:
            best = score
    return best


print(explain(best_score))

best_score:
  line 3: `for candidate in candidates` -- not split
      line 5: best is read before the iteration assigns it, so it carries a value from the previous iteration


`best` is read at the start of an iteration and holds what an earlier
iteration left. The fix is to do the expensive part inside the loop, collect,
and reduce afterwards. The reduction is cheap and runs once.

In [11]:
def best_score(candidates):
    scores = []
    for candidate in candidates:
        scores.append(-(candidate - 3.2) ** 2)
    return max(scores)


print(explain(best_score))

best_score:
  line 3: `for candidate in candidates` -- SPLIT
      collected in order: scores
      private to each iteration: candidate
  code before the loop runs in the caller
  code after the loop runs in the caller


A running total that is only ever added to needs no rewrite: `total += x`
is a collector. It is the *reading* of the carried value inside the loop that
prevents division, as in a running balance where each step multiplies the
previous value. When the dependence is real, as in a time-stepped simulation,
the loop cannot be divided; divide an outer loop instead (over parameter
sets, over seeds).

### `break`

In [12]:
def first_failures(samples, limit):
    failures = []
    for sample in samples:
        if len(failures) == limit:
            break
        if sample % 7 == 0:
            failures.append(sample)
    return failures


print(explain(first_failures))

first_failures:
  line 3: `for sample in samples` -- not split
      line 5: break stops the loop early, so later iterations depend on earlier ones
      line 7: failures is modified in the loop and also read in it, so an iteration can see what an earlier one did


Whether an iteration runs at all depends on the ones before it. If you can
afford to evaluate every item, do so and cut the list afterwards.

In [13]:
def first_failures(samples, limit):
    failures = []
    for sample in samples:
        if sample % 7 == 0:
            failures.append(sample)
    return failures[:limit]


print(explain(first_failures))

first_failures:
  line 3: `for sample in samples` -- SPLIT
      collected in order: failures
      private to each iteration: sample
  code before the loop runs in the caller
  code after the loop runs in the caller


### The global random number generator

In [14]:
def noisy_means(sizes):
    means = []
    for size in sizes:
        means.append(float(np.random.normal(size=size).mean()))
    return means


print(explain(noisy_means))

noisy_means:
  line 3: `for size in sizes` -- not split
      line 4: np.random.normal() draws from a random generator shared by all iterations, so each draw depends on the ones before it; create one per iteration instead, e.g. rng = np.random.default_rng(seed + i)


`np.random.normal` and `random.random` draw from one generator shared by the
whole process, so the numbers an iteration gets depend on how many were drawn
before it. Give each iteration its own generator. This is also what makes a
simulation reproducible replicate by replicate.

In [15]:
def noisy_means(sizes, seed=0):
    means = []
    for index, size in enumerate(sizes):
        rng = np.random.default_rng([seed, index])
        means.append(float(rng.normal(size=size).mean()))
    return means


print(explain(noisy_means))

noisy_means:
  line 3: `for (index, size) in enumerate(sizes)` -- SPLIT
      collected in order: means
      read-only (checked at run time): seed
      private to each iteration: index, rng, size
  code before the loop runs in the caller
  code after the loop runs in the caller


### A value used after the loop

In [16]:
def fit_all(settings):
    results = []
    for setting in settings:
        fitted = setting * 2.0
        results.append(fitted)
    return results, fitted


print(explain(fit_all))

fit_all:
  line 3: `for setting in settings` -- not split
      line 3: fitted is assigned in the loop and used after it, which depends on which iteration ran last


`fitted` after the loop means "whatever the last iteration assigned". Take it
from what was collected instead.

In [17]:
def fit_all(settings):
    results = []
    for setting in settings:
        fitted = setting * 2.0
        results.append(fitted)
    return results, results[-1]


print(explain(fit_all))

fit_all:
  line 3: `for setting in settings` -- SPLIT
      collected in order: results
      private to each iteration: fitted, setting
  code before the loop runs in the caller
  code after the loop runs in the caller


### A write to a global

In [18]:
PROCESSED = 0


def process(items):
    global PROCESSED
    outputs = []
    for item in items:
        PROCESSED += 1
        outputs.append(item.upper())
    return outputs


print(explain(process))

process:
  line 7: `for item in items` -- not split
      line 7: PROCESSED is a global or nonlocal the loop assigns
      line 8: PROCESSED is read before the iteration assigns it, so it carries a value from the previous iteration


A global assigned in the loop would be changed in the workers, not here.
Return the information instead, and update the global from the result if you
still need it.

In [19]:
def process(items):
    outputs = []
    for item in items:
        outputs.append(item.upper())
    return outputs, len(outputs)


print(explain(process))

process:
  line 3: `for item in items` -- SPLIT
      collected in order: outputs
      private to each iteration: item
  code before the loop runs in the caller
  code after the loop runs in the caller


### Loops clustrix does not look at

Only a `for` loop (or a single list, set or dict comprehension statement) at
the top level of the decorated function is considered. A loop nested under
`if` or `with`, a `while` loop, and a loop inside a helper the function calls
are not.

In [20]:
def guarded(items):
    outputs = []
    if items:
        for item in items:
            outputs.append(item * 2)
    return outputs


print(explain(guarded))

guarded:
  no loop at the top level of the function to split


Move the loop to the top level (here, by returning early for the empty case)
or decorate the helper that contains it. When a function has several loops
that qualify, the first one is divided and the rest run after it as written.

## Loops found unsafe while running

Reading the source can show that a loop is unsafe to divide. It cannot show
that one is safe, because it cannot see what the functions it calls do or
which objects two names refer to. Three checks therefore run with the real
values. If one fails, the division is abandoned, the original loop runs once
in order, and you get a warning. The answer is the ordinary one either way.

### Two names for the same object

In [21]:
@cluster(cores=4)
def growing(items):
    collected = []
    seen_so_far = collected          # a second name for the same list
    for item in items:
        collected.append(item + len(seen_so_far))
    return collected


print(explain(growing))

growing:
  line 5: `for item in items` -- SPLIT
      collected in order: collected
      read-only (checked at run time): seen_so_far
      private to each iteration: item
  code before the loop runs in the caller
  code after the loop runs in the caller


By the source, `collected` is only written and `seen_so_far` is only read, so
the loop passes. They are the same list, and each iteration reads its length.

In [22]:
print(growing([10, 20, 30, 40, 50, 60]))
print("plain loop:      ", [10 + 0, 20 + 1, 30 + 2, 40 + 3, 50 + 4, 60 + 5])

growing: not split -- seen_so_far can reach the same object as collected, so an iteration can see what earlier ones collected


[10, 21, 32, 43, 54, 65]
plain loop:       [10, 21, 32, 43, 54, 65]


The check ran before any work was handed out, the warning names both
variables, and the result is the sequential one.

### An argument changed by a helper

In [23]:
def remember(seen, record):
    seen.add(record)


@cluster(cores=4)
def running_distinct(records, seen):
    counts = []
    for record in records:
        remember(seen, record)
        counts.append(len(seen))
    return counts


print(explain(running_distinct))

running_distinct:
  line 8: `for record in records` -- SPLIT
      collected in order: counts
      read-only (checked at run time): seen
      private to each iteration: record
  code before the loop runs in the caller
  code after the loop runs in the caller


In [24]:
print(running_distinct([3, 1, 3, 2, 1, 4, 2, 5], set()))

running_distinct: not split -- the loop changed seen, which its iterations share


[1, 2, 2, 3, 3, 4, 4, 5]


Each chunk compares the values it reads before and after it runs. `seen` came
back changed, so the loop was run again in order. Note what that costs: the
iterations ran twice, once in the workers and once here. A loop that always
falls back should be rewritten or decorated with `parallel=False`. (Had the
body called `seen.add(record)` directly, the source alone would have been
enough to refuse it, with no wasted run.)

### Randomness hidden in a helper

In [25]:
import random


def jitter(value):
    return value + random.random()


@cluster(cores=4)
def jittered(values):
    out = []
    for value in values:
        out.append(jitter(value))
    return out


random.seed(42)
from_cluster = jittered([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])

random.seed(42)
plain = [value + random.random() for value in [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]]
print("same as a plain loop with the same seed:", from_cluster == plain)

jittered: not split -- the loop changed the global random number generator, which its iterations share


same as a plain loop with the same seed: True


## What no check can see

The checks cover the values the loop was handed: its arguments and the
variables it reads. They do not cover state that lives elsewhere:

- a module-level cache, counter or registry that a called function updates;
- files, databases and network services the iterations write to;
- a library's internal state, including the random state of a GPU.

Such side effects still happen once per iteration, but in the worker
processes (or the jobs), and in no particular order between chunks. Here is
the first kind made visible.

In [26]:
CALL_LOG = []


def expensive(x):
    CALL_LOG.append(x)        # a module-level side effect
    return x * x


@cluster(cores=4)
def squares(values):
    out = []
    for value in values:
        out.append(expensive(value))
    return out


print("result:           ", squares([1, 2, 3, 4, 5, 6]))
print("CALL_LOG, here:   ", CALL_LOG)

result:            [1, 4, 9, 16, 25, 36]
CALL_LOG, here:    []


The result is right and no warning was given. Whether `CALL_LOG` in this
process was filled depends on where the loop ran, which is exactly the kind
of thing your code should not rely on. If you need what an iteration
produced, have the loop collect it and return it. Dividing a loop is safe
when its iterations communicate only through what they return; that
assumption is yours to check, and clustrix does not claim to have proved it.

## Is dividing worth it? The first call measures

Handing work to other processes has a cost. For a loop that takes a
millisecond, the cost is larger than the work. Clustrix handles this by
measurement rather than by guessing:

- the **first** call of a decorated function divides its loop and measures,
  in the workers, how long an iteration takes;
- **later** calls divide only if the measured time per iteration, times the
  number of iterations, reaches a threshold;
- a call that runs undivided is timed too, so the decision follows the
  function if its work grows.

The threshold is a constant in the package.

In [27]:
from clustrix.decorator import MIN_SECONDS_WORTH_DIVIDING

print(MIN_SECONDS_WORTH_DIVIDING, "seconds")

0.2 seconds


A loop with a cheap body, called three times. The `INFO` log shows the
decision.

In [28]:
import logging

logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("clustrix").setLevel(logging.INFO)


@cluster(cores=4)
def cheap(numbers):
    pids = set()
    for number in numbers:
        pids.add(os.getpid())
    return sorted(pids)


for attempt in (1, 2, 3):
    pids = cheap(list(range(100)))
    print(f"call {attempt}: ran in", "this process" if pids == [os.getpid()] else f"{len(pids)} worker(s)")

cheap: its loop is too short to be worth dividing (about 0.001s in all); it runs in this process.


cheap: its loop is too short to be worth dividing (about 0.000s in all); it runs in this process.


call 1: ran in 4 worker(s)
call 2: ran in this process
call 3: ran in this process


The first call was divided and measured. The measurement said the whole loop
takes far less than the threshold, so the later calls ran here.

The measurement belongs to the function, not to its arguments. That has one
consequence worth knowing: if a function is called with a small problem and
then with a large one, the first large call runs undivided, is timed, and the
one after it is divided again.

In [29]:
small = fast_hash_chain(list(range(8)), 10)            # cheap iterations
_, pids = timed("large, right after small", fast_hash_chain, list(range(8)), 2_000_000)
print("   processes:", len(pids), "- this one" if pids == [os.getpid()] else "")
_, pids = timed("large again", fast_hash_chain, list(range(8)), 2_000_000)
print("   processes:", len(pids))

hash_chain: its loop is too short to be worth dividing (about 0.001s in all); it runs in this process.


large, right after small       2.61 s
   processes: 1 - this one


large again                    0.77 s
   processes: 4


In [30]:
logging.getLogger("clustrix").setLevel(logging.WARNING)

## Printing from a loop body

Output printed by the iterations is captured in the workers and shown here in
iteration order, once the loop has finished. It does not appear line by line
while the loop is running.

In [31]:
@cluster(cores=4)
def report(subjects):
    done = []
    for subject in subjects:
        print(f"subject {subject}: processed by {os.getpid()}")
        done.append(subject)
    return len(done)


report(["s01", "s02", "s03", "s04", "s05", "s06"])

subject s01: processed by 9002
subject s02: processed by 9002
subject s03: processed by 9001
subject s04: processed by 9001
subject s05: processed by 9003
subject s06: processed by 9002


6

For progress on a long loop, write to a log file from the iterations and watch
the file; printed output arrives only at the end.

## Exceptions

An exception raised in an iteration is raised here, as the same exception
type. The traceback from the worker is attached as its cause, so the failing
line of your function is visible. The remaining iterations are abandoned.

In [32]:
@cluster(cores=4)
def reciprocals(values):
    out = []
    for value in values:
        out.append(1.0 / value)
    return out


try:
    reciprocals([4.0, 2.0, 0.0, 1.0])
except ZeroDivisionError as error:
    print("caught:", repr(error))
    print()
    print("its cause:", type(error.__cause__).__name__)
    print(error.__cause__)

caught: ZeroDivisionError('float division by zero')

its cause: RemoteTraceback

Traceback (most recent call last):
  File ".../site-packages/clustrix/loop_workers.py", line 152, in worker_main
    outcome: Tuple[str, Any] = ("ok", function(*arguments))
                                      ~~~~~~~~^^^^^^^^^^^^
  File ".../site-packages/clustrix/loop_workers.py", line 122, in timed_call
    result = function(*arguments)
  File "<dynamic-code>", line 5, in __cx_body
ZeroDivisionError: float division by zero



`except ZeroDivisionError` caught it, as it would for a plain call. The cause
is the traceback as it was in the worker. In it your loop body appears as
`__cx_body` in `<dynamic-code>`: that is the function clustrix generated from
the body of your loop. If you do not catch the exception, Jupyter prints this
worker traceback first and its own below it.

## `cores=-1`, and other values

`cores=-1` means every CPU this process may use. Inside a SLURM allocation
that is the allocation (`SLURM_CPUS_PER_TASK`), not the whole node. Asking for
more workers than there are CPUs gives you as many workers as CPUs.

In [33]:
@cluster(cores=-1)
def spread(n):
    pids = set()
    for index in range(n):
        sum(i * i for i in range(200_000))
        pids.add(os.getpid())
    return len(pids)


print("worker processes used:", spread(32), "of", available_cpus(), "CPUs")

worker processes used: 4 of 4 CPUs


Other values that are not a positive whole number are refused when the
function is decorated.

In [34]:
for bad in (0, -2, 2.5, True):
    try:
        cluster(cores=bad)
    except ValueError as error:
        print(str(error)[:110])

@cluster(cores=0) is not a usable worker count: cores must be a positive integer, or -1 for every CPU availabl
@cluster(cores=-2) is not a usable worker count: cores must be a positive integer, or -1 for every CPU availab
@cluster(cores=2.5) is not a usable worker count: cores must be a positive integer, or -1 for every CPU availa
@cluster(cores=True) is not a usable worker count: cores must be a positive integer, and a bool is not one -- 


## Nested calls

A decorated function may call another decorated function inside its loop. The
outer loop is divided; the inner calls then run inside the workers and are not
divided again, so the machine is not oversubscribed.

In [35]:
@cluster
def inner(values):
    out = []
    for value in values:
        out.append((value, os.getpid()))
    return out


@cluster(cores=4)
def outer(groups):
    results = []
    for group in groups:
        results.append(inner(group))
    return results


for group_result in outer([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]]):
    print(group_result)

[(1, 9004), (2, 9004), (3, 9004)]
[(4, 9001), (5, 9001), (6, 9001)]
[(7, 9003), (8, 9003), (9, 9003)]
[(10, 9002), (11, 9002), (12, 9002)]


Each group was handled by one worker, and all of `inner`'s iterations for that
group ran in that same worker. (If `inner` asks for `cores` itself, each
worker logs that the request had no effect there.)

## Switching it off

`@cluster(parallel=False)` runs the function as one piece whatever its loop
looks like, and `configure(auto_parallel=False)` does that for every function.

In [36]:
_, pids = cluster(cores=4, parallel=False)(hash_chain)(list(range(4)), 1000)
print("ran in this process:", pids == [os.getpid()])

@cluster(cores=4) has no effect here: the local backend runs the decorated function once, in this process. Locally, cores is the number of worker processes a loop in the function is divided among; with no loop that can be divided, the function runs once, in this process. clustrix.explain(function) says why a loop was not divided.


ran in this process: True


With `parallel=False` and `cores` greater than one, clustrix warns that the
cores are not used. That is the message shown above.

## If something looks wrong

| You see | It means |
|---|---|
| `... has no loop that can be split` | `explain(function)` gives the line and reason. Rewrite as shown above, or accept one process. |
| `not split -- X can reach the same object as Y` | Two names refer to one object the loop writes to. The loop ran in order; the answer is right. |
| `not split -- the loop changed X, which its iterations share` | An iteration (or a function it called) modified a value the others read. The loop ran twice; rewrite it. |
| `not split -- the loop changed the global random number generator` | A helper draws from `random` or `np.random`. Pass each iteration its own generator. |
| A divided call is slower than the direct one | The loop is short, or its inputs are large and must be copied to each worker. Later calls adjust themselves; `parallel=False` turns division off. |
| The second call of a function runs in this process | The first call measured the loop as shorter than `MIN_SECONDS_WORTH_DIVIDING`. |
| Nothing is printed until the loop ends | Output from iterations is replayed in order at the end. |
| A module-level cache or counter is empty after the loop | It was updated in the workers. Return the data instead. |

## Next

[Tutorial 3](03_ssh_cluster.ipynb) sends the same kind of function to a remote
machine, where each chunk of the loop becomes a job.